In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
import numpy as np
import pandas as pd
import os
import random
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

import src.global_config as gcfg
import src.gnn.config as gnn_config
import src.stats.config as stat_cfg
import src.gnn.pipeline as pipeline
import src.gnn.analysis as eanalysis
import src.stats.analysis as stat_analysis

In [4]:
def ejecutar_30_corridas():
    num_corridas = 30
    resultados = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Iniciando {num_corridas} corridas en: {device}")

    # cargamos las metricas que ya calculamos
    df_stage2 = pd.read_csv(os.path.join(gcfg.OUTPUT_DIR, f"metricas_{stat_cfg.LEVEL}.csv"))
    
    # Nombres de las features originales para el alineamiento del PCA
    nombres_features = [f[1] for f in gcfg.FEATURES]
    df_features = eanalysis.obtener_promedios_anuales(df_stage2['year'].values, nombres_features, gcfg.DATA_FOLDER)

    for i in range(num_corridas):
        print(f"\nProcesando Corrida {i+1}/{num_corridas} | Seed: {i} ---")
        
        # semillas
        random.seed(i)
        np.random.seed(i)
        torch.manual_seed(i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(i)

        # Entrenar red neuronal
        historia = pipeline.run_temporal_graphsage(device)

        # Calcular PCA (1D) y Alinear
        df_flujo, metadatos_pca = eanalysis.calcular_pca_1d(historia)
        ev1_corrida = metadatos_pca["ev1"]
        w_corrida = metadatos_pca["loading"]

        # Diferencia entre embeddings  generados por medio del coseno
        if i == 0:
            # La primera corrida es nuestra referencia: el norte
            w_ref = w_corrida
            coseno_ref = 1.0 # Es idéntica a sí misma
        else:
            # Producto punto dividido por las magnitudes (fórmula del coseno)
            coseno_ref = np.dot(w_corrida, w_ref) / (np.linalg.norm(w_corrida) * np.linalg.norm(w_ref))
        
        df_flujo, _ = eanalysis.correlacionar_y_alinear(df_flujo, df_features, gnn_config.ALIGN_PC1_ANCHOR)
        df_flujo = df_flujo.rename(columns={"y_original": "sage_pc1"})

        # Unir datos y ajustar OLS
        df_all = pd.merge(df_stage2, df_flujo, on="year", how="inner")
        modelo_simple = stat_analysis.ajustar_modelo_ols(
            df_all, 
            y_col='sage_pc1', 
            x_cols=['mean_Percent of documents'], 
            usar_hac=True, 
            maxlags=3
        )

        # Pruebas de Hipótesis sobre los Residuos
        residuos = modelo_simple.resid
        
        # Normalidad
        _, shapiro_p = stats.shapiro(residuos)
        # Homocedasticidad (unicamente tomamos el valor del p value)
        _, bp_pvalue, _, _ = het_breuschpagan(residuos, modelo_simple.model.exog)
        # Independencia
        dw_stat = durbin_watson(residuos)

        print(f"  -> R2: {modelo_simple.rsquared:.4f} | Shapiro (p): {shapiro_p:.4f} | BP (p): {bp_pvalue:.4f} | DW: {dw_stat:.4f} | EV1: {ev1_corrida:.4f} | Coseno: {coseno_ref:.4f}")

        # Seleccion de features relevantes (del cual su coeficiente es distinto de cero)
        
        columnas_esperadas = [f"mean_{feat[1]}" for feat in gcfg.FEATURES] #unicamente agregamos  el prefijo mean_ 
        
        # un pequeño filtro, 
        todas_las_features = [col for col in columnas_esperadas if col in df_all.columns]
        features_candidatas = []
        
        
        # Filtramos por medio de regresion simple
        for feature in todas_las_features:
            mod_simple = stat_analysis.ajustar_modelo_ols(
                df_all, 'sage_pc1', [feature], usar_hac=True, maxlags=3
            )
            # Si el p-value de la feature es menor a 0.05, pasa a la final
            if mod_simple.pvalues[feature] < 0.05:
                features_candidatas.append(feature)

        # regresion multiple con los features relevantes
        if len(features_candidatas) > 0: # Solo armamos el modelo múltiple si al menos una feature pasó el filtro
            mod_multiple = stat_analysis.ajustar_modelo_ols(
                df_all, 'sage_pc1', features_candidatas, usar_hac=True, maxlags=3
            )
            
            # Ver quién sobrevivió al modelo múltiple (p < 0.05)
            sobrevivientes = []
            for feat in features_candidatas:
                if mod_multiple.pvalues[feat] < 0.05:
                    sobrevivientes.append(feat)
                    
            r2_final = mod_multiple.rsquared
            r2_adj_final = mod_multiple.rsquared_adj
            residuos = mod_multiple.resid
            
        else:
            # esto no deberia pasar pero por si acaso
            sobrevivientes = []
            r2_final = 0
            r2_adj_final = 0
            residuos = np.zeros(len(df_all)) # Residuos dummy para que no truene el código

        # algunas metricas sobre el modelo multiple
        _, shapiro_p = stats.shapiro(residuos) if len(features_candidatas) > 0 else (0, 0)
        
        if len(features_candidatas) > 0:
            _, bp_pvalue, _, _ = het_breuschpagan(residuos, mod_multiple.model.exog)
            dw_stat = durbin_watson(residuos)
        else:
            bp_pvalue, dw_stat = 0, 0

        # Imprimir progreso
        print(f"  -> R2 Múltiple: {r2_final:.4f} | Candidatas: {len(features_candidatas)} | Sobrevivientes: {len(sobrevivientes)}")

        # --- GUARDAR RESULTADOS ---
        resultados.append({
            "corrida": i + 1,
            "semilla": i,
            "r2_multiple": r2_final,
            "r2_adj_multiple": r2_adj_final,
            "num_candidatas": len(features_candidatas),
            "features_candidatas": " | ".join(features_candidatas), # Guardamos como texto separado por |
            "num_sobrevivientes": len(sobrevivientes),
            "features_sobrevivientes": " | ".join(sobrevivientes),
            "shapiro_p_value": shapiro_p,
            "breusch_pagan_p_value": bp_pvalue,
            "durbin_watson_stat": dw_stat,
            "pca_ev1": ev1_corrida,
            "pca_coseno_ref": coseno_ref
        })

    # uardar todo en CSVfg
    df_resultados = pd.DataFrame(resultados)
    nombre_dataset = os.path.basename(os.path.normpath(gcfg.DATA_FOLDER))
    out_csv = os.path.join(gcfg.OUTPUT_DIR, f"resultados_30_corridas_{nombre_dataset}.csv")
    df_resultados.to_csv(out_csv, index=False)
    
    print(f"\nLas 30 corridas terminaron. Resultados guardados en: {out_csv}")

if __name__ == "__main__":
    ejecutar_30_corridas()

Iniciando 30 corridas en: cpu
Calculando promedios anuales de los datos crudos (sin ego)...

Procesando Corrida 1/30 | Seed: 0 ---
Feature mode: ones (in_channels=1)
Warm-start: False
Modo: Sin memoria
Epoch 001 | Loss: 0.7843
Epoch 020 | Loss: 0.5211
Epoch 040 | Loss: 0.4581
Epoch 060 | Loss: 0.4218
1980 procesado. (alters=26, total_nodos=27)
Epoch 001 | Loss: 0.6980
Epoch 020 | Loss: 0.5025
Epoch 040 | Loss: 0.4441
Epoch 060 | Loss: 0.5192
1981 procesado. (alters=63, total_nodos=64)
Epoch 001 | Loss: 0.7594
Epoch 020 | Loss: 0.5298
Epoch 040 | Loss: 0.4576
Epoch 060 | Loss: 0.4677
1982 procesado. (alters=58, total_nodos=59)
Epoch 001 | Loss: 0.6797
Epoch 020 | Loss: 0.4642
Epoch 040 | Loss: 0.4490
Epoch 060 | Loss: 0.4691
1983 procesado. (alters=35, total_nodos=36)
Epoch 001 | Loss: 1.3994
Epoch 020 | Loss: 0.5149
Epoch 040 | Loss: 0.4376
Epoch 060 | Loss: 0.4346
1984 procesado. (alters=45, total_nodos=46)
Epoch 001 | Loss: 1.1448
Epoch 020 | Loss: 0.5039
Epoch 040 | Loss: 0.5144
Epo

KeyboardInterrupt: 

In [5]:

from pathlib import Path

nombre_dataset = os.path.basename(os.path.normpath(gcfg.DATA_FOLDER))
ruta_csv = os.path.join(gcfg.OUTPUT_DIR, f"resultados_30_corridas_{nombre_dataset}.csv")
df_res = pd.read_csv(ruta_csv)

total_corridas = len(df_res)

print(f"Resultados en base al dataset: {Path(gcfg.DATA_FOLDER).name}")

print("=== Resultados de las 30 corridas ===")

# El R-cuadrado
r2_mean = df_res['r2_multiple'].mean()
r2_std = df_res['r2_multiple'].std()
print(f"R^2 Promedio:        {r2_mean:.4f} ± {r2_std:.4f}")

# Significancia del Feature (Como es múltiple, vemos si sobrevivió al menos 1 variable)
beta_significativos = (df_res['num_sobrevivientes'] > 0).sum()
print(f"Veces que el modelo tuvo features significativos: {beta_significativos} de {total_corridas}")

print("\n=== Los supuestos ===")

# Normalidad (Shapiro-Wilk) -> Queremos p > 0.05
shapiro_pass = (df_res['shapiro_p_value'] > 0.05).sum()
print(f"Pasó Normalidad (Shapiro):           {shapiro_pass} de {total_corridas} corridas")

# Homocedasticidad (Breusch-Pagan) -> Queremos p > 0.05
bp_pass = (df_res['breusch_pagan_p_value'] > 0.05).sum()
print(f"Pasó Homocedasticidad (B-P):         {bp_pass} de {total_corridas} corridas")

# Independencia (Durbin-Watson) -> Queremos que el promedio ande cerca del 2
dw_mean = df_res['durbin_watson_stat'].mean()
dw_std = df_res['durbin_watson_stat'].std()
print(f"Durbin-Watson Promedio:              {dw_mean:.4f} ± {dw_std:.4f}")

print("\n=== Estabilidad geometrica de la generacion de embeddings (PCA) ===")

# Varianza Explicada (EV1) -> ¿Qué tan dominante es PC1?
ev1_mean = df_res['pca_ev1'].mean()
ev1_std = df_res['pca_ev1'].std()
print(f"Varianza Explicada (EV1) Promedio:   {ev1_mean:.4f} ± {ev1_std:.4f}")

# Similitud Angular (Coseno) -> ¿La red apunta siempre para el mismo lado?
cos_mean = df_res['pca_coseno_ref'].mean()
cos_std = df_res['pca_coseno_ref'].std()
print(f"Coseno Promedio vs Semilla 0:        {cos_mean:.4f} ± {cos_std:.4f}")

Resultados en base al dataset: SOM
=== Resultados de las 30 corridas ===
R^2 Promedio:        0.4548 ± 0.1743
Veces que el modelo tuvo features significativos: 22 de 30

=== Los supuestos ===
Pasó Normalidad (Shapiro):           13 de 30 corridas
Pasó Homocedasticidad (B-P):         28 de 30 corridas
Durbin-Watson Promedio:              1.6936 ± 0.4911

=== Estabilidad geometrica de la generacion de embeddings (PCA) ===
Varianza Explicada (EV1) Promedio:   0.7082 ± 0.1203
Coseno Promedio vs Semilla 0:        0.1764 ± 0.3019


In [6]:
nombre_dataset = os.path.basename(os.path.normpath(gcfg.DATA_FOLDER))
print(nombre_dataset)
print(os.path.normpath(gcfg.DATA_FOLDER))

SOM
Data/SOM


In [7]:
''' 
Resultados en base al dataset: BN
=== Resultado de las 30 corridas ===
R^2 Promedio:        0.8454 ± 0.0826
Veces que el modelo tuvo features significativos: 30 de 30

=== Los supuestos ===
Pasó Normalidad (Shapiro):           18 de 30 corridas
Pasó Homocedasticidad (B-P):         8 de 30 corridas
Durbin-Watson Promedio:              1.8056 ± 0.3398

=== Estabilidad geometrica de la generacion de embeddings (PCA)  ===
Varianza Explicada (EV1) Promedio:   0.6238 ± 0.0889
Coseno Promedio vs Semilla 0:        0.2476 ± 0.3004

'''

' \nResultados en base al dataset: BN\n=== Resultado de las 30 corridas ===\nR^2 Promedio:        0.8454 ± 0.0826\nVeces que el modelo tuvo features significativos: 30 de 30\n\n=== Los supuestos ===\nPasó Normalidad (Shapiro):           18 de 30 corridas\nPasó Homocedasticidad (B-P):         8 de 30 corridas\nDurbin-Watson Promedio:              1.8056 ± 0.3398\n\n=== Estabilidad geometrica de la generacion de embeddings (PCA)  ===\nVarianza Explicada (EV1) Promedio:   0.6238 ± 0.0889\nCoseno Promedio vs Semilla 0:        0.2476 ± 0.3004\n\n'

In [8]:
from collections import Counter

# Cargar el nuevo archivo de resultados
nombre_dataset = os.path.basename(os.path.normpath(gcfg.DATA_FOLDER))
ruta_csv = os.path.join(gcfg.OUTPUT_DIR, f"resultados_30_corridas_{nombre_dataset}.csv")
df_res = pd.read_csv(ruta_csv)

total_corridas = len(df_res)

print(f"=== Modelo multiple ({total_corridas} corridas) - dataset: {nombre_dataset} ===")

# El R-cuadrado Conjunto
r2_mean = df_res['r2_multiple'].mean()
r2_std = df_res['r2_multiple'].std()
print(f"R^2 Múltiple Promedio:       {r2_mean:.4f} ± {r2_std:.4f}")

print("\n=== Seleccion de features ===")
# Contar variables candidatas y sobrevivientes
contador_candidatas = Counter()
contador_sobrevivientes = Counter()

for _, row in df_res.iterrows():
    # Separar los strings por el delimitador ' | ' y limpiar espacios
    cands_str = str(row['features_candidatas'])
    sobres_str = str(row['features_sobrevivientes'])
    
    cands = [f.strip() for f in cands_str.split('|') if f.strip() and f.strip() != 'nan']
    sobres = [f.strip() for f in sobres_str.split('|') if f.strip() and f.strip() != 'nan']
    
    contador_candidatas.update(cands)
    contador_sobrevivientes.update(sobres)

print("Frecuencia de supervivencia en el modelo final (p < 0.05 conjunta):")
# Mostrar primero las variables que más sobrevivieron
for feature, conteo in contador_sobrevivientes.most_common():
    candidata_conteo = contador_candidatas[feature]
    porcentaje = (conteo / total_corridas) * 100
    print(f" - {feature}: Sobrevivió en {conteo}/{total_corridas} corridas ({porcentaje:.1f}%)")



=== Modelo multiple (30 corridas) - dataset: SOM ===
R^2 Múltiple Promedio:       0.4548 ± 0.1743

=== Seleccion de features ===
Frecuencia de supervivencia en el modelo final (p < 0.05 conjunta):
 - mean_Percent of documents: Sobrevivió en 12/30 corridas (40.0%)
 - mean_Ave. citations: Sobrevivió en 11/30 corridas (36.7%)
 - mean_Percent of documents Int. Coll.: Sobrevivió en 7/30 corridas (23.3%)
 - mean_Document Types: Sobrevivió en 6/30 corridas (20.0%)
 - mean_Documents,: Sobrevivió en 4/30 corridas (13.3%)
 - mean_Ave. authorships: Sobrevivió en 4/30 corridas (13.3%)
 - mean_WoS Categories: Sobrevivió en 3/30 corridas (10.0%)
